# The crowd table with holes: starter

Loads the data, fits one small model for the forecast holes (gradient boosting on what each unit showed up to `now`),
keeps the sample submission (the reference baselines, which score 1) for the other kinds, and writes `/kaggle/working/submission.csv`. It needs no internet.
Works on the camera table and on the H3 plaza tables alike: a row is a unit and a minute.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

INPUT = "/kaggle/input"      # the competition data (read only); no internet needed
OUT = "/kaggle/working"      # submission.csv goes here
DATA = next(r for r, _, f in sorted(os.walk(INPUT, followlinks=True))
            if {"train.csv", "test.csv", "sample_submission.csv"} <= set(f))
print("data:", DATA)
warnings.filterwarnings("ignore", category=RuntimeWarning)   # empty windows give NaN on purpose
train = pd.read_csv(os.path.join(DATA, "train.csv"))
test = pd.read_csv(os.path.join(DATA, "test.csv"))
sample = pd.read_csv(os.path.join(DATA, "sample_submission.csv"))
holes = test[test["id"].notna()].copy()
print(f"{len(train):,} training rows, {len(test) - len(holes):,} released test rows, {len(holes):,} holes")
print(holes["kind"].value_counts().to_string())

data: /kaggle/input/competitions/MDA-Ethical-AI-KAUST-Academy
13,443 training rows, 4,154 released test rows, 4,457 holes
kind
forecast         2660
hidden            849
hidden-future     484
missing           464


## One table per recording

Minutes down, units across, people where released (a hole or a blank minute is NaN).

In [2]:
def tables(df):
    df = df[df["people"].notna()]
    out = {}
    for rec, g in df.groupby("recording"):
        t = g.pivot_table(index="minute_of_day", columns="unit", values="people")
        out[rec] = t.reindex(np.arange(t.index.min(), t.index.max() + 1))
    return out

T_train, T_test = tables(train), tables(test)

## Features known at `now`

For each unit: its last value (5 minutes back at most), its mean over the last 15 minutes, the whole table's mean over
the last 5 minutes (all as logs) and the horizon. Nothing after `now`.

In [3]:
def features(t, now, H):
    a = t.loc[now - 15:now].to_numpy(float)
    last = np.full(a.shape[1], np.nan)
    for row in a[-6:]:
        last = np.where(np.isfinite(row), row, last)
    with np.errstate(all="ignore"):
        mean15 = np.nanmean(a, 0)
        table = np.nanmean(np.log1p(a[-6:]))
    X = pd.DataFrame({"log_last": np.log1p(last), "log_mean15": np.log1p(mean15), "log_table": table, "horizon": float(H)},
                     index=t.columns)
    return X

X, y = [], []
for rec, t in T_train.items():
    t.attrs["rec"] = rec
    for now in t.index[20:-5:3]:
        for H in (5, 15, 30):
            if now + H not in t.index:
                continue
            f = features(t, now, H)
            target = np.log1p(t.loc[now + H].to_numpy(float)) - f["log_last"].to_numpy()
            ok = np.isfinite(target)
            X.append(f[ok]); y.append(target[ok])
X, y = pd.concat(X), np.concatenate(y)
model = HistGradientBoostingRegressor(loss="absolute_error", max_iter=200, learning_rate=0.05, random_state=0)
model.fit(X, y, sample_weight=np.maximum(np.expm1(X["log_last"]), 1.0))
print(f"forecast model on {len(y):,} training rows")

forecast model on 12,145 training rows


## Predict the forecast holes, keep the reference for the rest

In [4]:
pred = dict(zip(sample["id"], sample["people"]))
fc = holes[holes["kind"] == "forecast"]
for (rec, now, H), g in fc.groupby(["recording", "now_minute_of_day", "horizon"]):
    t = T_test[rec]
    t.attrs["rec"] = rec
    f = features(t, int(now), int(H)).reindex(g["unit"])
    p = np.expm1(f["log_last"].to_numpy() + model.predict(f))
    for i, v in zip(g["id"], p):
        if np.isfinite(v):
            pred[i] = max(0.0, float(v))
submission = pd.DataFrame({"id": holes["id"], "people": [pred[i] for i in holes["id"]]})
submission.to_csv(os.path.join(OUT, "submission.csv"), index=False)
print(submission.shape)
submission.head()

(4457, 2)


,id,people
4,20260918_1100_cam_I_1404,29.799706
5,20260918_1100_cam_I_1405,29.799706
6,20260918_1100_cam_I_1406,29.799706
7,20260918_1100_cam_I_1407,29.799706
8,20260918_1100_cam_I_1408,29.799706
